In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 999)
pd.set_option('display.max_rows', 999)
pd.set_option("display.max_seq_items", None)

### Data Import and Sorting

In [ ]:
#This should eventually get replaced by the 2025 data (or any data) from AWS
data = pd.read_csv('historical_pitches_rnn_data.csv')
data.head(-20)

In [ ]:
def sort_statcast(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(["game_date", "game_pk", "inning", "inning_topbot", "at_bat_number", "pitch_number"], ascending=[True, True, True, False, True, True]).reset_index(drop=True)

In [ ]:
data = sort_statcast(data)
data.head(20)

In [ ]:
# We want to have a unique id for each plate appearance (PA)
data["pa_id"] = data["game_pk"].astype(str) + "_" + data["at_bat_number"].astype(str)

In [ ]:
# Getting rid of infrequently used pitches
pitch_remapping = {
    'SC': 'OTHER',
    'PO': 'OTHER',
    'CS': 'OTHER',
    'FA': 'OTHER',
}

data['pitch_type'] = data['pitch_type'].replace(pitch_remapping)

In [ ]:
# Handling automatic balls and strikes. Some entries have a pitch type of NaN. By looking at the description of the pitch,
#   we can see this is due to pitch clock violations. This replaces the NaNs with ABS instead.
abs_remapping = {
    'automatic_ball': 'ABS',
    'automatic_strike': 'ABS',
}
data['pitch_type'] = (data['description'].map(abs_remapping).fillna(data['pitch_type']))

In [ ]:
# Handling if previous pitch in sequence was ABS 
abs_label = ["ABS"]

is_abs = data["pitch_type"].isin(abs_label)
data["pitch_type_for_prev"] = data["pitch_type"].mask(is_abs)

# Track last real pitch type within each PA, will be NaN until first pitch is actually thrown
data["last_real_pitch_type"] = (data.groupby("pa_id")["pitch_type_for_prev"].ffill())

# Previous pitch type is the last real pitch but shifted 1. NaNs are now filled
data["prev_pitch_type"] = (data.groupby("pa_id")["last_real_pitch_type"].shift(1).fillna("START"))

# don't care about thse columns
data = data.drop(columns=["pitch_type_for_prev", "last_real_pitch_type"])


In [ ]:
# For each plate appearance, get the length (number of pitches thrown)
data["seq_len"] = data.groupby("pa_id")["pitch_type"].transform("size")

In [ ]:
data = data[data["game_type"] == 'R']
data = data[data["pitch_type"] != 'UN']

### Feature Creation

#### Rolling averages of certain stats

In [ ]:
# Creating Indicator columns 
# We should have these already but im just doing this based on the statcast set so making them again. 
swing_code = ['foul_bunt', 'foul', 'hit_into_play', 'swinging_strike', 'foul_tip',
                'swinging_strike_blocked', 'missed_bunt', 'bunt_foul_tip']
whiff_code = ['swinging_strike', 'foul_tip', 'swinging_strike_blocked']

# 0 = no, 1 = yes
data['is_swing'] = data['description'].isin(swing_code)
data['is_whiff'] = data['description'].isin(whiff_code)

data["is_called_strike"] = (data["description"] == "called_strike")
data["is_ball"] = data["description"].isin(["ball", "blocked_ball"])

data['in_zone'] = (data['zone'] < 10)
data['out_zone'] = (data['zone'] > 10)


#### Game State Features

In [ ]:
# Make base occupancy into bools
for base_col in ["on_1b", "on_2b", "on_3b"]:
    data[base_col] = data[base_col].notna()

# Make count a string (combining balls ans strikes)
data['count_state'] = (data['balls'].astype(int).astype(str) + "-" + data['strikes'].astype(int).astype(str))

data["score_diff_bat"] = data["bat_score"] - data["fld_score"]

data.head()

#### Dropping unneccessary columns

In [ ]:
drop_cols = [
    "player_name", "events", "sv_id", "umpire", "des",
    "spin_dir", "spin_rate_deprecated", "break_angle_deprecated", "break_length_deprecated", "game_type", "home_team", "away_team", "type", "hit_location", "bb_type", 
    "hc_x", "hc_y", "tfs_deprecated", "tfs_zulu_deprecated", "hit_distance_sc", "launch_speed", "launch_angle", "effective_speed", 
    "fielder_2", "fielder_3", "fielder_4",	"fielder_5", "fielder_6", "fielder_7", "fielder_8",	"fielder_9",
    "estimated_ba_using_speedangle", "estimated_woba_using_speedangle", "estimated_slg_using_speedangle", "woba_value", "woba_denom", "babip_value", "iso_value", "launch_speed_angle",
    "if_fielding_alignment", "of_fielding_alignment", "delta_home_win_exp", "hyper_speed", "bat_speed", "swing_length", "home_win_exp", "bat_win_exp", 
    "age_pit_legacy", "age_bat_legacy", "age_pit", "age_bat", "attack_angle", "attack_direction", "swing_path_tilt", 
    "intercept_ball_minus_batter_pos_x_inches", "intercept_ball_minus_batter_pos_y_inches", "Unnamed: 0", 
    "delta_run_exp", "delta_pitcher_run_exp", "batter_days_until_next_game", "api_break_z_with_gravity", "api_break_x_arm", "api_break_x_batter_in", "batter_days_until_next_game",
    "pitcher_days_until_next_game", "batter_days_since_prev_game", "pitcher_days_since_prev_game", "n_priorpa_thisgame_player_at_bat", "n_thruorder_pitcher", 
    # "is_whiff", "is_called_strike", "is_ball", "is_zone", "zone",
    "vx0", "vy0", "vz0", "ax", "ay", "az", "pfx_x", "pfx_z", "release_spin_rate", "spin_axis", "arm_angle", "plate_x", "plate_z", 'release_speed', 'release_pos_x', 'release_pos_z', 'release_extension', 'release_pos_y',
    # "at_bat_number", "pitch_number", "batter", "pitcher", "game_pk", "game_date", "description", "pitch_name", "game_year", "description"
    'post_away_score','post_home_score', 'post_bat_score', 'post_fld_score',
]
    
clean_data = data.drop(columns=[c for c in drop_cols], errors="ignore")
clean_data.head()

clean_data.info()

In [ ]:
clean_data.to_csv('rnn_data.csv', index=False)